# 07 - Customer Segmentation
## Unsupervised K-Means Clustering
**Objective:** Cluster customers into distinct behavioral cohorts using RFM and engagement features. Systematically evaluate multiple K values ($K=2..6$) using the Elbow Method and Silhouette Analysis without assuming $K=3$ upfront (Rule 13 compliant).


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

cust_path = os.path.join("..", "data", "processed", "customer_features.csv")
df = pd.read_csv(cust_path)


### 1. Feature Selection & Scaling


In [ ]:
cluster_features = [
    'recency_days',
    'total_sessions',
    'total_purchases',
    'total_revenue',
    'avg_pages_viewed',
    'cart_abandonment_rate'
]
X = df[cluster_features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


### 2. Multi-K Evaluation (Elbow & Silhouette)


In [ ]:
k_range = range(2, 7)
inertias = []
silhouettes = []
db_indices = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    db = davies_bouldin_score(X_scaled, labels)
    silhouettes.append(sil)
    db_indices.append(db)
    print(f"K={k}: Inertia={km.inertia_:.1f}, Silhouette={sil:.4f}, DB-Index={db:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_range, inertias, 'bo-', lw=2)
axes[0].set_title("Elbow Method (Inertia vs K)")
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("Inertia")

axes[1].plot(k_range, silhouettes, 'ro-', lw=2)
axes[1].set_title("Silhouette Score vs K")
axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()


### 3. Optimal Model Training (K = 3)


In [ ]:
best_k = 3
final_km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = final_km.fit_predict(X_scaled)

# Cluster summary profiles
profile = df.groupby('cluster').agg(
    Customer_Count=('customer_id', 'count'),
    Avg_Revenue=('total_revenue', 'mean'),
    Avg_Purchases=('total_purchases', 'mean'),
    Avg_Sessions=('total_sessions', 'mean'),
    Avg_Recency=('recency_days', 'mean'),
    Avg_Cart_Abandon=('cart_abandonment_rate', 'mean')
).reset_index()

# Map to business interpretations based on centroid stats
sorted_clusters = profile.sort_values(by='Avg_Revenue', ascending=False)['cluster'].tolist()
label_map = {
    sorted_clusters[0]: "Premium High-Value Customers",
    sorted_clusters[1]: "Regular Engaged Customers",
    sorted_clusters[2]: "Occasional Low-Engagement Customers"
}
profile['Segment_Name'] = profile['cluster'].map(label_map)
profile


### Conclusion
K-Means clustering with K=3 establishes 3 distinct actionable segments: Premium High-Value, Regular Engaged, and Occasional Low-Engagement Customers.
